In [ ]:
# ===================================================================================
# MIGRATION PRE-FLIGHT CHECK (Health Scanner) - ACTIONABLE + VTL STYLEURL FIX
# - Scans Web Maps for dependencies (operationalLayers + basemap + nested group layers).
# - Checks if Enterprise Feature Services already exist on Target (ledger + tag fallback).
# - Correctly handles VectorTileLayer "styleUrl" (common: AGOL vector tile reference layers)
#   so they DO NOT show as "UNKNOWN" blockers.
#
# OUTPUT:
#   1) READY vs BLOCKED maps
#   2) Missing Enterprise Service IDs (copy/paste into Feature Service migrator)
#   3) Unknown blockers (true unknowns that have neither url nor styleUrl)
#   4) External/AGOL informational list (validate access from target environment)
#
# READ ONLY: Does not create or modify any items.
# ===================================================================================

import pandas as pd
import os
import urllib3
import requests
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
requests.packages.urllib3.disable_warnings()

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC ---
# LIST THE WEB MAPS YOU WANT TO CHECK

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_preflight_webmaps.json")
if os.path.exists(_sidecar_path):
    import json as _json
    WEB_MAP_IDS_TO_CHECK = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(WEB_MAP_IDS_TO_CHECK)} Web Map IDs from sidecar.")
else:
    WEB_MAP_IDS_TO_CHECK = [
        # Example Source ID's
        # "ab93793e3e4f4adca4b4dc789f8e75d6",
    ]

# =============================================================================
# --- LOAD HISTORY + LEDGER ----------------------------------------------------
# =============================================================================
ALREADY_MIGRATED_IDS = set()
MIGRATION_LEDGER = {}  # (SourceID:str, LayerIndex:int) -> TargetID:str

def load_history_and_ledger():
    global ALREADY_MIGRATED_IDS, MIGRATION_LEDGER

    if not os.path.exists(LOG_FILE):
        print(f"ℹ No history file found at: {LOG_FILE}")
        return

    try:
        df = pd.read_csv(LOG_FILE)

        if "SourceID" in df.columns:
            ALREADY_MIGRATED_IDS = set(df["SourceID"].astype(str).str.strip())

        required = {"SourceID", "LayerIndex", "TargetID"}
        if required.issubset(set(df.columns)):
            for _, row in df.iterrows():
                sid = str(row.get("SourceID", "")).strip()
                tid = str(row.get("TargetID", "")).strip()
                idx = row.get("LayerIndex", "")

                if not sid or not tid:
                    continue
                try:
                    idx_int = int(idx)
                except:
                    continue

                MIGRATION_LEDGER[(sid, idx_int)] = tid

        print(f"Loaded history: {len(ALREADY_MIGRATED_IDS)} SourceIDs")
        print(f"Loaded ledger entries: {len(MIGRATION_LEDGER)} (SourceID,LayerIndex)->TargetID")
    except Exception as e:
        print(f"⚠ Could not read history/ledger: {e}")

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
print("Connecting for Scan...")
load_history_and_ledger()

source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL)
target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL)

print(f"   > Source: {source_gis.url}")
print(f"   > Target User: {target_gis.users.me.username}")

# =============================================================================
# --- HELPERS ------------------------------------------------------------------
# =============================================================================
def iter_layers_recursive(layer_list):
    """Yield every layer dict including nested group layers."""
    if not isinstance(layer_list, list):
        return
    for lyr in layer_list:
        if isinstance(lyr, dict):
            yield lyr
            if "layers" in lyr and isinstance(lyr["layers"], list):
                yield from iter_layers_recursive(lyr["layers"])

def infer_layer_index_from_url(url: str) -> int:
    """If URL ends with /<number>, treat it as sublayer index."""
    if not url:
        return 0
    try:
        last = url.split("/")[-1]
        if last.isdigit():
            return int(last)
    except:
        pass
    return 0

def find_source_item_by_url(layer_url: str):
    """
    Best-effort: maps a service URL back to a Source portal item id by searching source portal.
    Only works for items actually hosted/registered in the source portal.
    """
    if not layer_url:
        return None

    try:
        if "/MapServer" in layer_url:
            base = layer_url.split("/MapServer")[0] + "/MapServer"
            res = source_gis.content.search(f'url:"{base}"', max_items=1)
            if res:
                return res[0].id

        if "/FeatureServer" in layer_url:
            base = layer_url.split("/FeatureServer")[0] + "/FeatureServer"
            res = source_gis.content.search(f'url:"{base}"', max_items=1)
            if res:
                return res[0].id
    except:
        pass

    return None

def summarize_layer_keys(layer: dict):
    """Helpful debug keys if resolution fails."""
    keep = ["title", "id", "layerType", "type", "url", "itemId", "styleUrl", "visibility"]
    out = {}
    for k in keep:
        if k in layer:
            out[k] = layer.get(k)
    return out

def classify_layer(layer: dict):
    """
    Returns: (class, reason)
    class in {"enterprise", "agol", "external", "no_url"}
    FIX: VectorTileLayer often has no 'url' but does have 'styleUrl'.
    """
    style_url = (layer.get("styleUrl", "") or "").lower()
    if style_url:
        if "arcgis.com" in style_url or "arcgisonline.com" in style_url:
            return "agol", "VectorTileLayer styleUrl points to ArcGIS Online (non-migrated dependency)."
        return "external", "VectorTileLayer styleUrl points to external host."

    url = (layer.get("url", "") or "").lower()
    if not url:
        return "no_url", "Layer has no 'url' (and no styleUrl) — manual review needed."

    if "arcgisonline.com" in url or "services.arcgisonline.com" in url:
        return "agol", "ArcGIS Online / Living Atlas style URL."

    if ENTERPRISE_HOST in url:
        return "enterprise", "Enterprise-hosted service."

    return "external", "Non-enterprise external host."

def resolve_source_service_id(layer: dict):
    """
    Resolve Source enterprise service item id from layer JSON.
    Tries:
      1) layer['itemId']
      2) Source portal lookup by URL (only for enterprise-hosted services)
    """
    if "itemId" in layer and layer["itemId"]:
        return layer["itemId"]

    url = layer.get("url", "") or ""
    if url and (ENTERPRISE_HOST in url):
        return find_source_item_by_url(url)

    return None

def target_exists_for_enterprise_dependency(source_service_id: str, layer_index: int) -> bool:
    """
    Checks:
      1) Ledger (SourceID,LayerIndex)->TargetID and confirms target item exists
      2) Fallback tag search tags:"SourceID:<id>"
    """
    sid = str(source_service_id).strip()
    idx = int(layer_index)

    # 1) Ledger check
    t_id = MIGRATION_LEDGER.get((sid, idx))
    if t_id:
        try:
            if target_gis.content.get(t_id):
                return True
        except:
            pass

    # 2) Tag fallback (service-level)
    try:
        res = target_gis.content.search(f'tags:"SourceID:{sid}"', max_items=1)
        if res:
            return True
    except:
        pass

    return False

# =============================================================================
# --- MAIN ---------------------------------------------------------------------
# =============================================================================
print(f"\nScanning {len(WEB_MAP_IDS_TO_CHECK)} Web Maps for dependencies...\n")

maps_ready = []
maps_blocked = []

missing_enterprise_services = set()   # IDs to paste into feature service migrator
blocked_details = []                 # enterprise missing OR true unknowns OR external (if BLOCK_EXTERNAL_LAYERS)
external_notes = []                  # informational external layers (if not blocking)

for map_id in WEB_MAP_IDS_TO_CHECK:
    try:
        web_map = source_gis.content.get(map_id)
        if not web_map:
            print(f"❌ Map {map_id} not found.")
            maps_blocked.append(f"{map_id} (not found)")
            continue

        data = web_map.get_data()
        if not isinstance(data, dict):
            print(f"⚠ Map {map_id} returned non-dict data; skipping.")
            maps_blocked.append(f"{web_map.title} (bad data)")
            continue

        containers = []
        if "operationalLayers" in data:
            containers.append(("operationalLayers", data["operationalLayers"]))
        if "baseMap" in data and isinstance(data.get("baseMap"), dict) and "baseMapLayers" in data["baseMap"]:
            containers.append(("baseMapLayers", data["baseMap"]["baseMapLayers"]))

        map_blockers = []

        for container_name, layer_list in containers:
            for layer in iter_layers_recursive(layer_list):
                title = layer.get("title", "(no title)")
                layer_type = layer.get("layerType", layer.get("type", ""))
                url = layer.get("url", "") or ""
                idx = infer_layer_index_from_url(url)

                cls, cls_reason = classify_layer(layer)

                # External / AGOL handling
                if cls in ("agol", "external"):
                    note = {
                        "WebMapTitle": web_map.title,
                        "WebMapID": map_id,
                        "Container": container_name,
                        "LayerTitle": title,
                        "LayerType": layer_type,
                        "URL": url or layer.get("styleUrl", ""),
                        "Class": cls,
                        "Reason": cls_reason,
                        "Debug": summarize_layer_keys(layer),
                        "Action": "Usually OK to keep as-is; verify access from target environment."
                    }
                    if BLOCK_EXTERNAL_LAYERS:
                        map_blockers.append({
                            "WebMapTitle": web_map.title,
                            "WebMapID": map_id,
                            "Container": container_name,
                            "LayerTitle": title,
                            "LayerType": layer_type,
                            "SourceServiceID": "EXTERNAL",
                            "LayerIndex": idx,
                            "URL": note["URL"],
                            "Reason": f"External dependency blocked by policy (BLOCK_EXTERNAL_LAYERS=True). {cls_reason}",
                            "Debug": note["Debug"]
                        })
                    else:
                        external_notes.append(note)
                    continue

                # No URL and no styleUrl: true unknown -> blocker
                if cls == "no_url":
                    map_blockers.append({
                        "WebMapTitle": web_map.title,
                        "WebMapID": map_id,
                        "Container": container_name,
                        "LayerTitle": title,
                        "LayerType": layer_type,
                        "SourceServiceID": "UNKNOWN",
                        "LayerIndex": 0,
                        "URL": "",
                        "Reason": "Layer has no URL and no styleUrl; manual review needed (embedded/config-driven).",
                        "Debug": summarize_layer_keys(layer)
                    })
                    continue

                # Enterprise dependency must exist on target
                src_sid = resolve_source_service_id(layer)
                if not src_sid:
                    map_blockers.append({
                        "WebMapTitle": web_map.title,
                        "WebMapID": map_id,
                        "Container": container_name,
                        "LayerTitle": title,
                        "LayerType": layer_type,
                        "SourceServiceID": "UNKNOWN",
                        "LayerIndex": idx,
                        "URL": url,
                        "Reason": "Enterprise URL but could not resolve source itemId (no itemId and URL search failed).",
                        "Debug": summarize_layer_keys(layer)
                    })
                    continue

                if not target_exists_for_enterprise_dependency(src_sid, idx):
                    map_blockers.append({
                        "WebMapTitle": web_map.title,
                        "WebMapID": map_id,
                        "Container": container_name,
                        "LayerTitle": title,
                        "LayerType": layer_type,
                        "SourceServiceID": str(src_sid),
                        "LayerIndex": int(idx),
                        "URL": url,
                        "Reason": "Enterprise dependency missing on target (ledger + SourceID tag fallback failed).",
                        "Debug": summarize_layer_keys(layer)
                    })
                    missing_enterprise_services.add(str(src_sid))

        if map_blockers:
            print(f"🔴 BLOCKED: {web_map.title}")
            print(f"   Blockers found: {len(map_blockers)}")
            for b in map_blockers:
                print(f"   - {b['LayerTitle']} | {b['LayerType']} | SourceID={b['SourceServiceID']} | idx={b['LayerIndex']}")
                if b["URL"]:
                    print(f"     URL: {b['URL']}")
                print(f"     Reason: {b['Reason']}")
            maps_blocked.append(web_map.title)
            blocked_details.extend(map_blockers)
        else:
            print(f"🟢 READY:   {web_map.title}")
            maps_ready.append(web_map.title)

    except Exception as e:
        print(f"❌ Error scanning {map_id}: {e}")
        maps_blocked.append(f"{map_id} (error)")

# =============================================================================
# --- FINAL REPORT -------------------------------------------------------------
# =============================================================================
print("\n" + "=" * 60)
print("                     PRE-FLIGHT REPORT")
print("=" * 60)
print(f"✅ Maps Ready to Migrate:        {len(maps_ready)}")
print(f"🔴 Maps Blocked (needs action):  {len(maps_blocked)}")
print(f"⚠ Missing Enterprise Service IDs:{len(missing_enterprise_services)}")
print(f"ℹ External/AGOL layers noted:   {len(external_notes)}")
print("-" * 60)

# 1) Missing enterprise services: copy/paste into feature service migrator
if missing_enterprise_services:
    print("\n⚠️  MISSING ENTERPRISE FEATURE SERVICES (migrate these first):\n")
    print("MULTI_LAYER_IDS = [")
    for sid in sorted(missing_enterprise_services):
        try:
            item = source_gis.content.get(sid)
            name = item.title if item else "Unknown"
            print(f'    "{sid}",  # {name}')
        except:
            print(f'    "{sid}",')
    print("]")

# 2) True unknown blockers: print debug info
unknown_blockers = [b for b in blocked_details if b["SourceServiceID"] in ("UNKNOWN", "EXTERNAL")]
if unknown_blockers:
    print("\n🧩 BLOCKERS DETAIL (manual review):\n")
    for b in unknown_blockers:
        print(f"- Map: {b['WebMapTitle']} ({b['WebMapID']})")
        print(f"  Layer: {b['LayerTitle']} | Type: {b['LayerType']} | idx={b['LayerIndex']}")
        if b["URL"]:
            print(f"  URL/Style: {b['URL']}")
        print(f"  Reason: {b['Reason']}")
        print(f"  Debug keys: {b['Debug']}\n")

# 3) External/AGOL informational report
if external_notes and not BLOCK_EXTERNAL_LAYERS:
    print("\n🌐 EXTERNAL / AGOL LAYERS (usually OK to keep, validate access):\n")
    for r in external_notes[:100]:
        print(f"- Map: {r['WebMapTitle']}")
        print(f"  Layer: {r['LayerTitle']} | Type: {r['LayerType']} | Class: {r['Class']}")
        print(f"  URL/Style: {r['URL']}")
        print(f"  Note: {r['Action']}\n")
    if len(external_notes) > 100:
        print(f"... (+{len(external_notes) - 100} more external layers)\n")

if maps_blocked:
    print("🛑 Some maps are blocked. Resolve blockers above before migrating those maps.")
else:
    print("🚀 All systems go! No blockers detected for the scanned maps.")

print("=" * 60)

# --- ORCHESTRATOR OUTPUT (write missing IDs for downstream consumption) ---
import json as _json
_output_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_output_preflight_webmaps.json")
with open(_output_path, "w", encoding="utf-8") as _f:
    _json.dump({"missing_ids": list(missing_enterprise_services)}, _f, indent=2)
print(f"[Orchestrator] Wrote {len(missing_enterprise_services)} missing FS IDs to {os.path.basename(_output_path)}")